# MobileNetV2 spot-check (CIFAR-10) --- magnitude only, 0.95 / 0.999

A minimal existence check, not a tuned sweep: does the BaCP effect appear on
a skip-connection architecture that is genuinely different from the
ResNet/VGG grid (inverted residuals, entirely depthwise-separable convs,
6x fewer parameters than resnet34), or is it specific to the architectures
already tested?

`FAMILIES['mobilenet_v2']` in `nb_common.py` reuses the ResNet/VGG recipe
by default and is explicitly flagged **UNVERIFIED** for this architecture --
VGG needed its own learning-rate fix after measurement
(`sec:experiments:vgglr`), and no equivalent measurement exists here. Treat
any result from this notebook as a spot-check of whether the effect appears,
not as a paper-ready number.

Scope, deliberately cut down from the full grid to keep this fast:
magnitude criterion only (no SNIP/WANDA), sparsities 0.95 and 0.999 (the
two ends of the grid, not the middle), seed 1 only. 5 training runs total:
1 dense + 2 I.P. + 2 BaCP.

Local CPU integration verified before writing this notebook: generic
head-swap dispatch (`.classifier`), CIFAR-scale stem adapter
(`adapt_mobilenet_for_small_images`), and magnitude pruning reaching exact
target sparsity across all 17 depthwise/grouped conv layers -- see the
commit adding `mobilenet_v2` to `model_factory.py` / `nb_common.py`.


In [ ]:
import sys, pathlib

# Find nb_common.py whether the kernel started in this folder or at the repo root.
here = pathlib.Path.cwd()
for cand in [here, *here.parents]:
    if (cand / 'nb_common.py').exists():
        sys.path.insert(0, str(cand)); break
    if (cand / 'project' / 'test_notebooks' / 'nb_common.py').exists():
        sys.path.insert(0, str(cand / 'project' / 'test_notebooks')); break
else:
    raise RuntimeError('cannot find nb_common.py -- start the kernel inside the repo')

import nb_common as nb
info = nb.setup()


## Configure

`SMOKE=True` runs every cell below on 2 batches first -- do that once on a
new cluster before real training.


In [ ]:
MODEL      = 'mobilenet_v2'
SEED       = 1
GPU        = 0
SPARSITIES = (0.95, 0.999)
SMOKE      = False
OVERRIDES  = {}       # e.g. dict(epochs=3) to shorten every run below

for phase in ('dense', 'prune', 'bacp'):
    print(f'{phase:>6}: ', {**nb.FAMILIES[MODEL]['base'], **nb.FAMILIES[MODEL][phase]})


## Weights + preflight

Halts before any GPU time is spent if the model is not actually pretrained
(`load_weights` fails soft, so a missing checkpoint would otherwise silently
train from random init -- this cost a day of GPU time once already).


In [ ]:
nb.fetch_imagenet_weights(MODEL)
nb.preflight(MODEL, num_classes=nb.FAMILIES[MODEL]['base']['num_classes'])


## Dense baseline (required first)

Both sparse arms below start from this checkpoint (same seed). Re-running
skips it if its record exists; delete the record under `results/runs/` to
re-arm.


In [ ]:
dense = nb.make_cell(MODEL, 'dense', seed=SEED, smoke=SMOKE, **OVERRIDES)
out = nb.run(dense, gpu=GPU)


## I.P. --- magnitude (Han et al. 2015)

Iterative pruning + recovery, the matched-budget baseline. One run per
sparsity level, streamed back to back.


In [ ]:
ip_cells = [nb.make_cell(MODEL, 'prune', seed=SEED, pruner='magnitude', sparsity=s,
                        smoke=SMOKE, **OVERRIDES) for s in SPARSITIES]
nb.run_group(ip_cells, gpu=GPU)


## BaCP --- magnitude

The contrastive objective (PrC/SnC/FiC + CE, lambdas 0.25 each, tau 0.15),
then the AdamW finetune.


In [ ]:
bacp_cells = [nb.make_cell(MODEL, 'bacp', seed=SEED, pruner='magnitude', sparsity=s,
                          smoke=SMOKE, **OVERRIDES) for s in SPARSITIES]
nb.run_group(bacp_cells, gpu=GPU)


## Verdict

No published MobileNetV2 row exists to compare against; `results_table`
still prints ours (paper columns come back `-`). What matters here is
`d(bacp - ip)` at each sparsity, printed directly below.


In [ ]:
nb.results_table(MODEL)

import json, glob, os
root = os.environ['BACP_RESULTS_DIR']
acc = {}
for f in glob.glob(os.path.join(root, 'runs', '*.json')):
    r = json.load(open(f, encoding='utf-8'))
    k = r.get('experiment_group') or ''
    if r.get('status') == 'ok' and '.smoke' not in k:
        acc[k] = r.get('test_acc_pct')

print()
print(f'{"sparsity":>9}  {"I.P.":>7} {"BaCP":>7}   {"delta":>7}')
for s in SPARSITIES:
    ip = acc.get(f'static.prune.{MODEL}.cifar10.s{s}.magnitude.seed{SEED}')
    bc = acc.get(f'static.bacp.{MODEL}.cifar10.s{s}.magnitude.seed{SEED}')
    f = lambda v: f'{v:7.2f}' if v is not None else '      -'
    d = lambda a, b: f'{a-b:+7.2f}' if (a is not None and b is not None) else '      -'
    print(f'{s:>9}  {f(ip)} {f(bc)}   {d(bc,ip)}')

print()
print('If BaCP beats I.P. at both sparsities, the effect generalises past')
print('ResNet/VGG to a genuinely different skip-connection topology -- worth')
print('a sentence in the paper as a spot-check, not a tuned result. If it')
print('does not, that is also worth stating: the effect may be architecture-')
print('dependent, same open question VGG-11 already raised.')


## Health

Every static record for this model. Delete a record to re-arm its cell.


In [ ]:
import runner as R
done = sorted(k for k in R.completed_keys() if k.startswith('static.') and MODEL in k)
print(f'{len(done)} static record(s) for {MODEL}:')
for k in done:
    print(' ', k)
